## 第一节: 数据的统一性(Data Uniformity)


### 1. 核心理念：为什么要统一？
数据一致性是指确保同一列数据具有相同的单位、格式和标准。 如果数据不统一（例如，有些以美元计，有些以欧元计），直接汇总计算（如“.sum（）”或“.mean（）”）会得出完全错误的结果。

---

### 2. 常见情形与治疗方案

#### A. 单位统一性
* **情景**：温度（摄氏度对华氏度）、体重（公斤对磅）。
**提示**：使用布尔索引查找需要修改的行并进行数学转换。
**代码模板**：

 ```python

 # 找到所有华氏行
 temp_f = df['temp_unit'] == 'F'
 # 转换并重写
 df.loc[temp_f，“温度”] = （df.loc[temp_f，“温度”] - 32） * * （5/9）
 df.loc[temp_f， 'temp_unit'] = 'C'
 ```

#### B. 货币统一性
* **情景**：不同货币的混合。
**代码模板**：
 ```python
 # 寻找欧元银行
 acct_eu = 银行业['acct_cur'] == '欧元'
 # 汇率转换与统一标签
 banking.loc[acct_eu， 'acct_amount'] = banking.loc[acct_eu， 'acct_amount'] * 1.1
 banking.loc[acct_eu， 'acct_cur'] = 'Dollar'
 ```

#### C. 日期一致性
**情景**：日期格式混乱（例如“2026-03-01”与“01/03/2026”）。
* **核心工具**：“pd.to_datetime（）”
* **扩展补充（高级参数）**：
   *  `dayfirst=True`：强制'01/03/2026'将解决至3月1日（欧洲习俗）。
   *  `yearfirst=True`：年份先行。
   *  `errors='coerce`：如果遇到无法解析的混乱日期，将其转换为“NaT”（非时间）以避免崩溃。
   *  `format='%Y-%m-%d`：手动指定格式，最快。

---

## 3. 核心工具：.loc 的“狙击手”模式
这是本章最重要的操作逻辑。

语法：`df.loc[row_indexer， column_indexer] = new_value`

`row_indexer（行索引）`：通常传递一个布尔序列（例如 'df['col'] > 10'）。
`column_indexer（列索引）`：指定要修改的列名。
**优势**：直接在原始表内存中修改，避免“SettingWithCopyWarning”警告，效率极高。

---

## 4. 补充扩展：跨领域验证
一致性是单列的整齐性，下一个“跨字段验证”是对多列的逻辑检查：
* **示例**：“存款+投资=总资产”？
* **Action**： `inconsistent_rows = df['total'] ！= (df['inv'] + df['dep'])`

---
## 第二节: 跨字段验证(cross field validation)

* **1. 核心思想：数据查账**
    * 跨字段验证是指利用同一行数据中不同字段（列）之间的逻辑关系来识别错误。

    * 单一验证：检查这一列数字是不是大于 0。

    * 跨字段验证：检查这几列数字加起来，是不是等于总计那一列。
---
* **2. 常用场景与代码套路**
    * A. 数值加总验证 (Sum Consistency)
    场景：总投资额 (total_inv) 必须等于 各分项投资 (inv_1, inv_2...) 之和。
    ```python
    # 1. 计算分项之和 (注意使用 axis=1 进行横向求和)
    sum_inv = banking[['inv_1', 'inv_2', 'inv_3']].sum(axis=1)

    # 2. 找出不一致的行
    inconsistent_rows = sum_inv != banking['total_inv']

    # 3. 过滤出这些“坏账”数据查看
    banking[inconsistent_rows]
    ```
    * B. 日期逻辑验证 (Date Logic)
    场景：利用出生日期 (birth_date) 和当前日期计算年龄，验证填写的年龄 (age) 是否准确。
    ```python
    import datetime as dt

    # 1. 获取今年年份
    today = dt.date.today()

    # 2. 计算理论年龄 (使用 .dt 访问器提取年份)
    # 只有 datetime64 类型才能使用 .dt
    ages_manual = today.year - banking['birth_date'].dt.year

    # 3. 找出虚报年龄的行
    age_diffs = ages_manual != banking['age']
    banking[age_diffs]
    ```
---
* **3. 重点工具解析**

| 工具 | 作用 | 备注 |
| :--- | :--- | :--- |
| **`.sum(axis=1)`** | **横向求和** | 默认是纵向(`axis=0`)，如果要算每一行的分项之和，必须设为 1 |
| **`.dt`** | **日期访问器** | 核心前提：该列必须已经是 `datetime64` 格式 |
| **`.dt.year`** | **提取年份** | 同理还有 `.dt.month` (月), `.dt.day` (日), `.dt.weekday` (周几) |
| **`.all()`** | **检查全集** | 判断序列中是否“每一个”都为 True，常用于 `assert` 自动化验证 |
---
* **4. 补充与拓展 (职业级技巧)**
  * 1. 处理浮点数精度问题 (np.isclose)
    在处理金额（小数）时，直接用 == 可能因为计算机二进制精度问题判定为不等。
    
    推荐写法：

    ```python
    import numpy as np
    # 只要差值极小，就认为它们相等
    is_equal = np.isclose(sum_inv, banking['total_inv'], atol=0.01)
    ```
  * 2. 发现错误后的处理决策
    一旦发现跨字段验证失败，通常有三种方式：

    `删除 (Drop)`：df.drop(df[inconsistent_rows].index)。

    `置空 (Set to NaN)`：df.loc[inconsistent_rows, 'age'] = np.nan。

    `标记 (Flag)`：创建一个新列 df['is_valid']，让后续分析人员知情。

  ---
### 第 3 节：数据完整性与缺失值可视化 (Completeness)

#### 1. 什么是缺失值？
缺失值是指在观测过程中没有存储任何数据值。
* **常见表现**：Pandas 中通常显示为 `NaN`（Not a Number）或 `NA`。
* **特殊陷阱**：有时会以 `0`、点 `.` 或特定数值（如 `-1`）的形式伪装出现。

---

#### 2. 缺失值的“文字版照妖镜”
在画图之前，先用代码摸清家底：
```python
# 1. 查看整张表每个格子是否有缺失 (返回 True/False)
banking.isna()

# 2. 【最常用】统计每一列具体缺了多少个值
banking.isna().sum()
```

---

#### 3. 核心大招：缺失值可视化 (missingno 实战)
文字统计只能看数量，但无法看出**缺失的规律**。这时候就需要引入 `missingno` 库了（简写为 `msno`）。

> *如果 VS Code 环境里还没安装，在终端跑一下 `pip install missingno`。*

##### A. 基础 X 光片：矩阵图 (Matrix)
最常用的图，能一眼看出整张表的缺失分布。

**实操代码：**
```python
import missingno as msno
import matplotlib.pyplot as plt

# 传入 DataFrame 画出矩阵图
msno.matrix(airquality)
plt.show()
```
![alt text](images/msno_matrix.png)

**🔍 图例与解读技巧（怎么看这幅图？）：**
1. **主图区域**：
   * **深色/黑色**：代表健康、**有数据**的格子。
   * **白色横线**：代表**缺失数据** (`NaN`)。如果看到某几列的白线是平行的，说明这几列的数据经常**同时丢失**（比如问卷调查里跳过了一整页）。
2. **右侧的微型折线图 (Sparkline)**：
   * 这是数据的“生命体征线”。它展示的是**每一行的完整度**。
   * 线条靠右、数字是满的（比如 5），说明这一行所有字段都填满了。
   * 线条往左凹陷、数字变小（比如 3），说明这一行缺了 2 个字段的数据。

##### B. 进阶玩法：寻找“假随机”的作案动机
如果直接画图，白线看起来是乱七八糟的（像**MCAR 完全随机**）。但如果怀疑 CO2 数据的缺失跟当天的气温（`Temperature`）有关，可以**先排序，再画图**！

**实操代码：**
```python
# 1. 先把整个表格按照 'Temperature' (气温) 从低到高排序
airquality_sorted = airquality.sort_values('Temperature')

# 2. 对排序后的表格画矩阵图
msno.matrix(airquality_sorted)
plt.show()
```
![alt text](images/msno_sorted.png)

**🔍 图例与解读技巧（排序图怎么看？）：**
* **观察聚集效应**：由于数据已经按气温从低到高排列（顶部是最冷的行，底部是最热的行），如果发现图表顶部的 `CO2` 列全是密集的白色横线，这就实锤了！
* **结论**：CO2 的缺失**不是随机的 (MAR 随机缺失)**，它就是在气温极低的时候集体罢工了！找到这个原因，你才能决定是去修传感器，还是用特定的低温公式来填补这些缺失值。

---

#### 4. 处理策略：删还是填？

##### 方案 A：直接丢弃 (Drop)
* **适用场景**：缺失比例极小，不影响整体分析大局。
```python
# 删除 'CO2' 这一列里有缺失值的所有行
airquality_cleaned = airquality.dropna(subset=['CO2'])
```

##### 方案 B：补全填坑 (Impute)
* **适用场景**：数据很珍贵，舍不得删，且缺失规律比较简单。
```python
# 计算 CO2 列的平均值
co2_mean = airquality['CO2'].mean()

# 用这个平均值把 CO2 列里的 NaN 全填上
airquality_imputed = airquality.fillna({'CO2': co2_mean})
```

### 缺失类型定义及含义
* **1. MCAR (完全随机缺失)**
  * 全称：Missing Completely At Random

  * 代表什么：数据缺失纯属偶然，没有任何规律，跟表格里的任何数据都无关。

  * 助记：有个 Completely（完全），意思就是“完全看老天爷掷骰子”，纯意外（比如一阵妖风刮跑了问卷）。

* **2. MAR (随机缺失)**
  * 全称：Missing At Random

  * 代表什么：缺失看似随机，其实有迹可循。它的缺失是由表格里的其他已知字段造成的（比如因为“气温太低”，导致“CO2没测到”）。

  * 助记：少了 Completely，说明它“并不完全随机”。有内鬼，查查旁边的列就能找到背后的原因。

* **1. MNAR (非随机缺失)**
  * 全称：Missing Not At Random

  * 代表什么：最棘手的一种。数据之所以缺失，恰恰是因为那个没填上去的数据本身（比如因为“客户满意度极低”，所以“拒绝填写满意度得分”）。

  * 助记：直接加了个 Not（不），明确告诉你“这绝对不是意外，是做贼心虚故意藏起来的”。